# SPM 3405: Machine Learning — Lecture 2 notebook
## Linear regression

**Dr Charles Ndung'u** · Department of Physics, University of Nairobi

In this notebook you will:
1. Calibrate a temperature sensor with least squares, and find the **standard errors**, **$t$-values** and **$R^2$**.
2. Fit a solar panel's power with **multiple linear regression**, a **dummy variable** for dust, and an **interaction**.
3. Turn a curve into a straight line: a **capacitor discharge**.
4. Write your own **gradient descent**, and see why the step size and scaling matter.

Reading: ISLP Chapter 3 (3.1–3.3). Run each cell with **Shift + Enter**, from top to bottom.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(405)   # fixed seed: everyone gets the same numbers as the slides

### A helper: least squares with standard errors

`fit(X, y)` returns the coefficients $\hat{\boldsymbol\beta}$, their standard errors, $t$-values and $p$-values, the RSE and $R^2$.
`X` must include a first column of ones (for the intercept). The formulas are the ones on the slides:
$\hat{\boldsymbol\beta}$ solves $\mathbf X^{\mathsf T}\mathbf X\,\hat{\boldsymbol\beta} = \mathbf X^{\mathsf T}\mathbf y$, and
$\mathrm{SE}(\hat\beta_j)^2$ is the $j$-th diagonal entry of $\hat\sigma^2(\mathbf X^{\mathsf T}\mathbf X)^{-1}$.

In [ ]:
def fit(X, y):
    """Least squares with SE, t and p for each coefficient. X must include a column of ones."""
    n, k = X.shape                                   # k = number of coefficients
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)     # solves the normal equations
    res = y - X @ beta                               # residuals
    rss = res @ res
    sigma2 = rss / (n - k)                           # estimate of the noise variance
    se = np.sqrt(np.diag(sigma2 * np.linalg.inv(X.T @ X)))
    t = beta / se
    p = 2 * stats.t.sf(np.abs(t), n - k)
    r2 = 1 - rss / np.sum((y - y.mean())**2)
    return beta, se, t, p, np.sqrt(sigma2), r2

def show(names, result):
    beta, se, t, p, rse, r2 = result
    print(f"{'':>10} {'value':>10} {'SE':>9} {'t':>8} {'p':>8}")
    for nm, b, s, tt, pp in zip(names, beta, se, t, p):
        print(f"{nm:>10} {b:10.4f} {s:9.4f} {tt:8.2f} {pp:8.4f}")
    print(f"RSE = {rse:.2f},  R^2 = {r2:.4f}")

## Part 1: Calibrating a temperature sensor

An analogue temperature sensor outputs a voltage $V$ (mV). The datasheet says $V = 10\,T$: a sensitivity of 10 mV/°C and no offset.
We record the sensor output at 20 temperatures measured by a reference thermometer. The sensor has noise with $\sigma = 15$ mV.

In [ ]:
n = 20
T_ref = np.sort(rng.uniform(5, 95, n))            # reference thermometer (deg C)
V = 10.0 * T_ref + rng.normal(0, 15, n)           # sensor output (mV)

X1 = np.c_[np.ones(n), T_ref]                     # column of ones, then T
cal = fit(X1, V)
show(["offset", "slope"], cal)

b0, b1 = cal[0]
plt.plot(T_ref, V, "o", label="readings")
grid = np.linspace(0, 100, 2)
plt.plot(grid, b0 + b1 * grid, label=f"V = {b0:.1f} + {b1:.2f} T")
plt.xlabel("reference temperature T (°C)"); plt.ylabel("sensor output V (mV)"); plt.legend(); plt.show()

The same slope from the formula on the slides, $\hat\beta_1 = \sum(x_i-\bar x)(y_i-\bar y) / \sum(x_i-\bar x)^2$:

In [ ]:
xb, yb = T_ref.mean(), V.mean()
slope = np.sum((T_ref - xb) * (V - yb)) / np.sum((T_ref - xb)**2)
print("slope     =", round(slope, 3), "mV/°C")
print("intercept =", round(yb - slope * xb, 2), "mV")
print("95% interval for the slope:", round(b1 - 2*cal[1][1], 2), "to", round(b1 + 2*cal[1][1], 2))
print("A reading of 500 mV means T =", round((500 - b0) / b1, 1), "°C")

**Questions**
1. Is the datasheet sensitivity (10 mV/°C) consistent with the data? Use the 95% interval.
2. Does the sensor have a real offset? Use the $t$-value and $p$-value of the offset.
3. Change the noise from 15 to 50 mV and run Part 1 again. What happens to the SE, the $t$-values and $R^2$?
4. Take readings only between 40 and 60 °C (change `rng.uniform(5, 95, n)`). What happens to the SE of the slope? Why?

## Part 2: A solar panel — many inputs

40 readings of a panel: irradiance $G$ (W/m²), cell temperature $T$ (°C), a dust indicator $D$ (1 = dusty) and power $P$ (W).
The panel gets hotter in strong sunlight, so $G$ and $T$ are correlated.

In [ ]:
m = 40
G = rng.uniform(200, 1000, m)                      # irradiance (W/m^2)
Tc = 20 + 0.03 * G + rng.normal(0, 4, m)           # cell temperature (deg C)
D = rng.integers(0, 2, m)                          # 1 = dusty panel, 0 = clean
P = 0.25 * G * (1 - 0.004 * (Tc - 25)) * (1 - 0.08 * D) + rng.normal(0, 3, m)

print("first rows (G, T, D, P):\n", np.round(np.c_[G, Tc, D, P][:3], 1))
print("correlation between G and T:", round(np.corrcoef(G, Tc)[0, 1], 2))

print("\nP ~ T alone")
show(["intercept", "T"], fit(np.c_[np.ones(m), Tc], P))
print("\nP ~ G + T")
show(["intercept", "G", "T"], fit(np.c_[np.ones(m), G, Tc], P))

**Question 5.** The coefficient of $T$ is **positive** alone and **negative** with $G$ included. Explain why.

### Dummy variable and interaction

In [ ]:
print("P ~ G + T + D")
show(["intercept", "G", "T", "D"], fit(np.c_[np.ones(m), G, Tc, D], P))
print("\nP ~ G + T + D + G*T")
show(["intercept", "G", "T", "D", "G*T"], fit(np.c_[np.ones(m), G, Tc, D, G * Tc], P))

In [ ]:
res = P - np.c_[np.ones(m), G, Tc] @ fit(np.c_[np.ones(m), G, Tc], P)[0]
plt.scatter(G, res, c=D, cmap="coolwarm")
plt.axhline(0, color="k", lw=0.8)
plt.xlabel("G (W/m²)"); plt.ylabel("residual (W)"); plt.title("Residuals of P ~ G + T (red = dusty)"); plt.show()

**Questions**
6. How much power does dust cost, with $G$ and $T$ held fixed? How sure are you?
7. Look at the residual plot of the model **without** $D$. How could you have spotted that an input was missing?
8. The average power of dusty panels is only a few watts below that of clean ones (`P[D==1].mean()`, `P[D==0].mean()`),
   yet the model says dust costs about 12 W. Explain.

## Part 3: Capacitor discharge — making a curve straight

A 100 µF capacitor discharges through 22 kΩ, so $\tau = RC = 2.2$ s and $V = V_0 e^{-t/\tau}$.
Taking logs gives a straight line: $\ln V = \ln V_0 - t/\tau$.

In [ ]:
t = np.linspace(0, 8, 17)                          # time (s)
Vc = 5.0 * np.exp(-t / 2.2) + rng.normal(0, 0.02, t.size)   # measured voltage (V)

cap = fit(np.c_[np.ones(t.size), t], np.log(Vc))
show(["ln V0", "slope"], cap)
print("tau =", round(-1 / cap[0][1], 3), "s;  V0 =", round(np.exp(cap[0][0]), 3), "V")
print("R^2 of a straight line on V itself:", round(fit(np.c_[np.ones(t.size), t], Vc)[5], 3))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(t, Vc, "o"); ax[0].set_xlabel("t (s)"); ax[0].set_ylabel("V (V)")
ax[1].plot(t, np.log(Vc), "o"); ax[1].plot(t, cap[0][0] + cap[0][1] * t)
ax[1].set_xlabel("t (s)"); ax[1].set_ylabel("ln V"); plt.tight_layout(); plt.show()

**Question 9.** The last few points on the $\ln V$ plot scatter more than the first ones. Why? (Hint: the noise is 0.02 V at every point.)

## Part 4: Gradient descent

Cost: $J(\boldsymbol\beta) = \frac{1}{2n}\sum_i (y_i - \hat y_i)^2$. Batch update:
$\boldsymbol\beta \leftarrow \boldsymbol\beta + \frac{\alpha}{n}\mathbf X^{\mathsf T}(\mathbf y - \mathbf X\boldsymbol\beta)$.

In [ ]:
def gradient_descent(X, y, alpha, steps):
    beta = np.zeros(X.shape[1])
    J = []
    for _ in range(steps + 1):
        r = y - X @ beta                           # residuals
        J.append(r @ r / (2 * len(y)))             # cost
        beta = beta + alpha / len(y) * X.T @ r     # step downhill
    return beta, np.array(J)

# standardise the temperature first
z = (T_ref - T_ref.mean()) / T_ref.std()
Xz = np.c_[np.ones(n), z]

for alpha in [0.01, 0.5, 2.1]:
    beta, J = gradient_descent(Xz, V, alpha, 40)
    plt.semilogy(J, label=f"alpha = {alpha}")
plt.xlabel("step"); plt.ylabel("cost J (mV²)"); plt.legend(); plt.show()

beta, J = gradient_descent(Xz, V, 0.5, 40)
slope = beta[1] / T_ref.std()
print("slope =", round(slope, 3), "mV/°C;  offset =", round(beta[0] - slope * T_ref.mean(), 2), "mV   (compare Part 1)")

In [ ]:
# without scaling: only a tiny step size is stable
alpha_max = 2 / np.linalg.eigvalsh(X1.T @ X1 / n).max()
beta_raw, J_raw = gradient_descent(X1, V, 0.9 * alpha_max, 5000)
print("largest stable alpha on raw data:", alpha_max)
print("after 5000 steps: offset =", round(beta_raw[0], 2), "mV, slope =", round(beta_raw[1], 3), "mV/°C")

**Questions**
10. Which $\alpha$ is too small, which is about right, and which is too large? How can you tell from the plot?
11. Why does gradient descent on the **raw** temperatures still have the wrong offset after 5000 steps?
12. **Your group project:** which inputs and output would a linear regression use for your problem? Would you need a dummy variable, an interaction or a transformation?

## Your turn: group tasks

Work on these in your group. Write the code in the empty cells and answer each question in a text cell below it,
in your own words. Submit **one notebook per group** on Google Classroom, with every cell run.
Every member must be able to explain every line.

The tasks reuse `fit`, `show` and `gradient_descent` from above, so run the whole notebook first.

### Task 1: A sensor that is not quite linear

A second temperature sensor is calibrated at 25 reference temperatures. Its output bends slightly at high temperature.
The cell below makes the data; do not change it.

In [ ]:
rng2 = np.random.default_rng(4052)                # task data: do not change
n2 = 25
T2 = np.sort(rng2.uniform(0, 100, n2))             # reference temperature (deg C)
V2 = 400 + 12 * T2 - 0.03 * T2**2 + rng2.normal(0, 10, n2)   # sensor output (mV)

(a) Fit the straight line $V = \beta_0 + \beta_1 T$ with `fit`. Plot the **residuals** against $T$.
What pattern do you see, and what does it tell you?

(b) Add a column $T^2$ and fit $V = \beta_0 + \beta_1 T + \beta_2 T^2$. Is the $T^2$ term needed? Use its $t$-value and $p$-value,
and compare the RSE and $R^2$ of the two models.

(c) The sensor reads 1000 mV. What temperature is that according to each model? (For the quadratic, solve
$\hat\beta_2 T^2 + \hat\beta_1 T + \hat\beta_0 - 1000 = 0$ with `np.roots` and keep the root between 0 and 100 °C.)
How large is the difference, in °C?

In [ ]:
# Task 1: your code here

### Task 2: Which solar-panel model predicts best?

In Lecture 1 we chose models by their **test** error. Here you do the same for the solar panel of Part 2.

(a) Split the 40 panel readings at random into 30 for training and 10 for testing (use `rng2.permutation(m)`).

(b) Fit each model below on the **training** readings only, and compute its training MSE and its test MSE:
`P ~ G`, `P ~ G + T`, `P ~ G + T + D`, `P ~ G + T + D + G*T`.
Print the four results as a table.

(c) Which model would you choose, and why? Does the model with the most inputs always have the lowest test MSE?

In [ ]:
# Task 2: your code here

### Task 3: Gradient descent with three inputs

(a) Standardise $G$, $T$ and $D$ (subtract the mean, divide by the standard deviation), and build a matrix with a column of ones and
the three standardised inputs.

(b) Run `gradient_descent` on all 40 readings with $\alpha = 0.1$, $0.5$ and $1.0$ for 200 steps. Plot the three cost curves.

(c) Check your answer: fit the same standardised model with `fit` and compare the coefficients. Then convert the gradient-descent
coefficient of $G$ back to watts per W/m² (divide by the standard deviation of $G$) and compare it with Part 2.

In [ ]:
# Task 3: your code here